In [ ]:
!pip install jsonlines

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
dataset_dir = "drive/MyDrive/chatbot_uhn"
os.chdir(dataset_dir)
save_dir = "response_uhn_chatbot_baru"
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive


In [ ]:
import pandas as pd

# Specify the path to your CSV file
csv_file_path = 'pertanyaan.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_file_path, header=0)

# Display the first few rows of the DataFrame to verify
print(df.head())

                                       Pertanyaan  \
0  Kapan sih Prodi Informatika ini mulai berdiri?   
1            Bagaimana sejarah prodi informatika?   
2          Dimasa saja tempat kuliah atau kampus?   
3           Apakah ada kelas online atau offline?   
4                    Apakah boleh memilih kampus?   

                                             Jawaban  
0  Prodi informatika Universitas Hindu Negeri I G...  
1  Prodi informatika Universitas Hindu Negeri I G...  
2  Prodi Informatika memiliki dua kampus yaitu di...  
3  Prodi Informatika memiliki dua kampus yaitu di...  
4  Prodi Informatika memiliki dua kampus yaitu di...  


In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

# model_transformers = "sentence-transformers/all-MiniLM-L6-v2"
# model_transformers = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_transformers = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
transformers_model_similarity = SentenceTransformer(model_transformers)

def transformers_get_context(question):
  # Get embeddings
  filtered_embedding = transformers_model_similarity.encode(df['Pertanyaan'], convert_to_tensor=True)
  query_embedding = transformers_model_similarity.encode(question, convert_to_tensor=True)

  # Compute cosine similarities
  cos_scores = util.cos_sim(query_embedding, filtered_embedding)[0]

  # Get top 1 score and index
  top_score, top_idx = torch.topk(cos_scores, k=1)

  # print("Top score:", top_score.item())
  # print("Top index:", top_idx.item())

  if top_score.item() * 100 > 50:
    return df.iloc[top_idx.item()]["Pertanyaan"], top_score.item(), df.iloc[top_idx.item()]["Jawaban"]
  else:
    return df.iloc[0]["Pertanyaan"], top_score.item(), df.iloc[0]["Jawaban"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

new_max_token = 256
chat_model_id = "aisingapore/Llama-SEA-LION-v3.5-8B-R"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_id)

chat_model = AutoModelForCausalLM.from_pretrained(
                  chat_model_id,
                  torch_dtype=torch.bfloat16
              ).to(DEVICE)

def chatbot_respond(message):
  input_ids = chat_tokenizer.apply_chat_template(
      message,
      add_generation_prompt=True,
      tokenize=False,
      thinking_mode="off"
  )

  inputs = chat_tokenizer(input_ids, return_tensors="pt")

  input_id = inputs["input_ids"].to(DEVICE)
  attention_mask=inputs["attention_mask"].to(DEVICE)

  outputs = chat_model.generate(
          input_id,
          attention_mask=attention_mask,
          max_new_tokens=new_max_token,
          pad_token_id=chat_tokenizer.eos_token_id,
          temperature=0.3,
          do_sample=True,
          repetition_penalty=1.1
      )

  answer = chat_tokenizer.decode(outputs[0], skip_special_tokens=True)

  return {"choices": [{"message": {"role": "assistant", "content": answer}}]}

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/840 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
import jsonlines
import os
from datetime import date, datetime

def save_json(save_dir, question, similar_question, similar_score,
              model_similarity, targeted_response, model_response, complete_response, time_response, username):

    save_file = os.path.join(save_dir, str(date.today())+ '.jsonl')
    os.makedirs(os.path.dirname(save_file), exist_ok=True)
    now = datetime.now()
    time = now.strftime("%Y:%m:%d %H:%M:%S")

    with jsonlines.open(save_file, mode='a') as writer:
        writer.write(
        {
            "time": time,
            "question": question,
            "similar_question": similar_question,
            "similar_score": similar_score,
            "model_similarity": model_similarity,
            "targeted_response": targeted_response,
            "model_response": model_response,
            "complete_response": complete_response,
            "time_response": str(time_response) + " seconds",
            "username": username
        }
    )

In [ ]:
import gradio as gr
import time
import threading
from datetime import datetime

def respond(user_message, history, username):
    # If history is None (first turn), initialize as empty list
    if history is None:
        history = []

    # Show the user message and an empty bot message first
    history = history + [{"role": "user", "content": f"<span style='font-size:0.85em; color: #888;'>{username}</span><br>{user_message}"}, {"role": "assistant", "content": ""}]
    yield history, "", gr.update(visible=False), gr.update(interactive=False)  # Clear input, show user message instantly

    # get the start time
    start_time = datetime.now()
    similar_question, similarity_score, description_detailed = transformers_get_context(user_message)

    content = f"Ini adalah jawaban anda tentang deskripsi \
        \"{description_detailed}\". Saya adalah pengunjung pada Program Studi Informatika kampus UHN I Gusti Bagus Sugriwa \
        ini adalah pertanyaan saya: \"{user_message}\"? Buatlah pendapat yang singkat, sopan dan bersahabat \
        untuk saya dengan gaya bahasa berusia 18 tahun dan jangan memberikan pertanyaan lanjutan. Jawablah dalam 50 kata."

    # similar_question, similarity_score, description_detailed = "", "", ""
    # content = user_message

    messages = [
        {"role": "user", "content": content},
    ]

    response = {}
    def infer():
        bot_response = chatbot_respond(messages)
        response["text"] = bot_response['choices'][0]['message']['content'].split("assistant\n\n")[-1].replace("<think>\n\n</think>\n\n", "")
        # get the end time
        end_time = datetime.now()
        time_difference = end_time - start_time
        seconds = int(time_difference.total_seconds())
        try:
            save_json(
                save_dir=save_dir,
                question=user_message,
                similar_question=similar_question,
                similar_score=similarity_score,
                model_similarity=model_transformers,
                targeted_response=description_detailed,
                model_response=response["text"],
                complete_response=bot_response,
                time_response=seconds,
                username=username)
        except:
            print("cannot save json files")
    t = threading.Thread(target=infer)
    t.start()

    # While waiting for model, animate "Bot is typing..."
    i = 0
    while t.is_alive():
        dots = '.' * (i % 10)
        history[-1]["content"] = f"Bot is typing{dots}"
        yield history, "", gr.update(visible=False), gr.update(interactive=False)
        time.sleep(0.3)
        i += 1

    bot_message = response["text"]
    words = bot_message.split()
    partial = ""
    for w in words:
        partial += w + " "
        history[-1]["content"] = f"<span style='font-size:0.85em; color:#888;'>Chatbot</span><br>{partial.strip()}"
        yield history, "", gr.update(visible=False), gr.update(interactive=False)   # Update chat window with each word
        time.sleep(0.2)     # Simulate delay for effect
    yield history, "", gr.update(visible=True), gr.update(interactive=False)

def enable_chat(username):
    # Enable chat if username is non-empty after stripping whitespace
    return gr.update(interactive=bool(username.strip()))

with gr.Blocks() as demo:
    gr.HTML("""
        <div style="display: flex; justify-content: space-between; align-items: center; line-height: 80%; margin-top: -10px;">
            <div style="margin-top: -10px; margin-bottom: -10px">
                <h2 style="display: flex; align-items: center;">
                  <img src="https://cdn-icons-png.flaticon.com/512/4712/4712035.png" width="50px" style="vertical-align: middle; margin-right: 12px;" />
                  Selamat datang <span style='color: #3b82f6; margin-left: 10px;'> ChatBot Prodi Informatika</span>
                </h2>
                <p>
                    <b>Tips:</b>
                    <ul style="margin-top: -8px;">
                        <li>Silahkan ketikkan pertanyaan tentang Prodi Informatika, UHN I Gusti Bagus Sugriwa.</li>
                        <li>ChatBot Prodi Informatika dengan senang hati akan membantu anda memberikan informasi!</li>
                    </ul>
                </p>
            </div>
            <div>
                <img src="https://upload.wikimedia.org/wikipedia/id/1/19/Logo_Universitas_Hindu_Negeri_I_Gusti_Bagus_Sugriwa.png" width="100px" style="margin-left: 14px;">
            </div>
        </div>
    """)
    username = gr.Textbox(
            show_label=False,
            placeholder="Masukkan nama anda untuk berdiskusi!"
        )
    chatbot = gr.Chatbot(
        label="Informatika - UHN I Gusti Bagus Sugriwa chatbot",
        avatar_images=(
            "https://cdn-icons-png.flaticon.com/512/1946/1946429.png",  # User avatar
            "https://cdn-icons-png.flaticon.com/512/4712/4712035.png"   # Bot avatar
        ),
        show_label=True,  # Optional: shows a label above the chatbox
        type="messages",
        value=[
            {"role": "assistant", "content": f"<span style='font-size:0.85em; color:#888;'>Chatbot</span><br>👋 Halo! selamat datang di Prodi Informatika, UHN I Gusti Bagus Sugriwa. Ketik nama anda sebelum memulai berdiskusi! Ada yang bisa saya bantu?"}
        ]
    )
    with gr.Row() as input_row:
        msg = gr.Textbox(show_label=False,
                         placeholder="Ketik pertanyaan anda",
                         interactive=False)
        submit_btn = gr.Button("💬 Kirim",
                               elem_id="send_btn",
                               scale=0)
    gr.HTML("""
    <style>
    #send_btn {
        height: 62px !important;
    }
    </style>
    """)

    # Enable chat input only if username is filled
    username.change(enable_chat, inputs=username, outputs=msg)

    msg.submit(respond, [msg, chatbot, username], [chatbot, msg, input_row, username])
    submit_btn.click(respond, [msg, chatbot, username], [chatbot, msg, input_row, username])

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8a13c240957f759311.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
